In [2]:
from mistralai import Mistral
from dotenv import load_dotenv
import datauri
import os

In [4]:
from dotenv import load_dotenv
import os
import subprocess

# Load environment variables from .bashrc
def load_bashrc_env(bashrc_path=os.path.expanduser("~/.bashrc")):
    # Use a login shell to source .bashrc and print env
    command = f"bash -c 'source {bashrc_path} >/dev/null 2>&1; env'"
    proc = subprocess.Popen(command, stdout=subprocess.PIPE, shell=True, executable="/bin/bash")
    for line in proc.stdout:
        (key, _, value) = line.decode().partition("=")
        os.environ[key.strip()] = value.strip()
    proc.communicate()

load_bashrc_env()
api_key = os.environ.get("MISTRAL_API_KEY")
if not api_key:
    raise EnvironmentError(
        "MISTRAL_API_KEY environment variable not set. "
        "Set it in your environment or in a .env file, e.g.:\n"
        "MISTRAL_API_KEY=your_api_key_here"
    )
client = Mistral(api_key=api_key)

In [5]:
def save_image(image):
    parsed = datauri.parse(image.image_base64)
    with open(image.id, "wb") as file:
      file.write(parsed.data)
      
def create_markdown_file(ocr_response, output_filename = "../assets/waris_grammar_sketch.md"):
  with open(output_filename, "wt") as f:
    for page in ocr_response.pages:
      f.write(page.markdown)
      for image in page.images:
        save_image(image)
        
def upload_pdf(filename):
  uploaded_pdf = client.files.upload(
    file={
      "file_name": filename,
      "content": open(filename, "rb"),
    },
    purpose="ocr"
  )
  signed_url = client.files.get_signed_url(file_id=uploaded_pdf.id)
  return signed_url.url

filename = "../assets/waris_grammar_sketch.pdf"
ocr_response = client.ocr.process(
  model="mistral-ocr-latest",
  document={
    "type": "document_url",
    "document_url": upload_pdf(filename),
  },
  include_image_base64=True,
)

create_markdown_file(ocr_response)